<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/Implementation_of_VisAnomReasoner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# [CELL 1 - Environment Setup]
!pip install -q torch torchvision transformers peft accelerate pillow matplotlib scipy scikit-learn

import os
import io
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.ensemble import IsolationForest

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

Executing on device: cuda


In [10]:
# [CELL 2 - Plot Renderer & Synthetic Benchmark Generator]

def generate_visanom_sample(seq_len=200, img_size=(400, 200)):
    """
    Generates a time-series signal, injects a localized anomaly,
    renders it as a PIL Image plot, and produces structured rationale annotations.
    """
    t = np.arange(seq_len)
    # Base signal: sine wave + random Gaussian noise
    base_signal = np.sin(2 * np.pi * t / 30.0) + np.random.normal(0, 0.15, size=seq_len)

    # Randomly select an anomaly interval [t_start, t_end]
    anomaly_len = np.random.randint(15, 25)
    t_start = np.random.randint(30, seq_len - anomaly_len - 30)
    t_end = t_start + anomaly_len

    anomaly_type = random.choice(["inverted_spike", "variance_shift", "level_shift"])
    signal = base_signal.copy()

    if anomaly_type == "inverted_spike":
        signal[t_start:t_end] = -3.0 * np.sin(2 * np.pi * t[t_start:t_end] / 30.0)
        desc = f"inverted spike where expected peaks flip sharply to negative troughs."
    elif anomaly_type == "variance_shift":
        signal[t_start:t_end] += np.random.normal(0, 1.2, size=anomaly_len)
        desc = f"sudden high-frequency variance shift with high noise amplitude."
    else: # level_shift
        signal[t_start:t_end] += 2.5
        desc = f"sustained positive baseline shift off the mean."

    # Render time-series to image plot
    fig, ax = plt.subplots(figsize=(img_size[0]/100, img_size[1]/100), dpi=100)
    ax.plot(t, signal, color='#1f77b4', linewidth=1.5)
    ax.set_xlim(0, seq_len)
    ax.set_ylim(-4.5, 4.5)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel("Time Index (t)", fontsize=8)
    ax.set_ylabel("Value", fontsize=8)
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=100)
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf).convert('RGB')

    # Formulate step-by-step natural language reasoning chain
    rationale = (
        f"Step 1: The plot displays a sequential signal across {seq_len} time units.\n"
        f"Step 2: A noticeable structural deviation occurs between t={t_start} and t={t_end}, showing an {desc}\n"
        f"Step 3: Anomaly located at interval [{t_start}, {t_end}]."
    )

    norm_interval = [t_start / seq_len, t_end / seq_len]
    return img, signal, (t_start, t_end), norm_interval, rationale

# Generate a verification sample
sample_img, sample_sig, raw_bounds, norm_bounds, sample_rationale = generate_visanom_sample()
print(f"Generated Sample Anomaly Bounds: {raw_bounds} (Normalized: {norm_bounds})")
print("Rationale Ground Truth:\n" + sample_rationale)

Generated Sample Anomaly Bounds: (119, 135) (Normalized: [0.595, 0.675])
Rationale Ground Truth:
Step 1: The plot displays a sequential signal across 200 time units.
Step 2: A noticeable structural deviation occurs between t=119 and t=135, showing an sustained positive baseline shift off the mean.
Step 3: Anomaly located at interval [119, 135].


In [11]:
# [CELL 3 - VisAnomBench Dataset Loader]

class VisAnomDataset(Dataset):
    def __init__(self, num_samples=300, seq_len=200):
        self.samples = [generate_visanom_sample(seq_len=seq_len) for _ in range(num_samples)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, signal, raw_bounds, norm_bounds, rationale = self.samples[idx]

        # Convert image to RGB Tensor
        img_arr = np.array(img).transpose(2, 0, 1) / 255.0
        img_tensor = torch.tensor(img_arr, dtype=torch.float32)
        signal_tensor = torch.tensor(signal, dtype=torch.float32)
        bounds_tensor = torch.tensor(norm_bounds, dtype=torch.float32)

        return {
            "image": img_tensor,
            "signal": signal_tensor,
            "bounds": bounds_tensor,
            "raw_bounds": raw_bounds,
            "rationale": rationale
        }

train_dataset = VisAnomDataset(num_samples=400)
test_dataset  = VisAnomDataset(num_samples=100)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Dataset Initialized: Train={len(train_dataset)} | Test={len(test_dataset)}")

Dataset Initialized: Train=400 | Test=100


In [15]:
# [CELL 4 - Fixed VisAnomReasoner with Positional Embeddings & Dual Heads]

class TemporalPreservingEncoder(nn.Module):
    """
    Pools vertical Y-axis values while preserving 1D X-axis temporal sequence structure.
    Adds learned 1D positional embeddings across time bins.
    """
    def __init__(self, embed_dim=128, seq_tokens=50):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),  # [32, 100, 200]
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # [64, 50, 100]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, embed_dim, kernel_size=3, stride=2, padding=1), # [128, 25, 50]
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(),
        )
        # Explicit 1D temporal positional embedding
        self.pos_embed = nn.Parameter(torch.randn(1, seq_tokens, embed_dim) * 0.02)

    def forward(self, x):
        feat = self.stem(x)               # [B, C, 25, 50]
        feat_1d = feat.mean(dim=2)        # Pool Y-axis -> [B, C, 50]
        feat_1d = feat_1d.transpose(1, 2) # [B, 50, C]
        return feat_1d + self.pos_embed   # Add learned 1D temporal positional embeddings

class VisAnomReasoner(nn.Module):
    """
    Dual-head Vision-Language Reasoner:
    1. Heatmap Head: Classifies each of the 50 temporal bins (Dense Supervision).
    2. Interval Head: Regresses precise normalized start/end bounds [t_start, t_end].
    """
    def __init__(self, embed_dim=128, seq_tokens=50):
        super().__init__()
        self.seq_tokens = seq_tokens
        self.visual_encoder = TemporalPreservingEncoder(embed_dim=embed_dim, seq_tokens=seq_tokens)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 1. Per-Bin Temporal Heatmap Head (Dense local supervision)
        self.heatmap_head = nn.Linear(embed_dim, 1)

        # 2. Global Interval Regression Head (Flattened temporal sequence -> [center, width])
        self.interval_head = nn.Sequential(
            nn.Linear(seq_tokens * embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 2),
            nn.Sigmoid()
        )

    def forward(self, images):
        tokens = self.visual_encoder(images)  # [B, 50, 128]
        fused = self.transformer(tokens)       # [B, 50, 128]

        # Dense Per-Bin Heatmap Logits [B, 50]
        heatmap_logits = self.heatmap_head(fused).squeeze(-1)

        # Flattened sequence regression preserving temporal order [B, 50 * 128]
        flat_feat = fused.reshape(fused.size(0), -1)
        raw_out = self.interval_head(flat_feat)       # [B, 2] -> [center, width]

        cx = raw_out[:, 0]
        w  = raw_out[:, 1] * 0.40 + 0.02              # Constrain width reasonably

        t_start = torch.clamp(cx - w / 2.0, min=0.0, max=1.0)
        t_end   = torch.clamp(cx + w / 2.0, min=0.0, max=1.0)

        pred_bounds = torch.stack([t_start, t_end], dim=-1)
        return pred_bounds, heatmap_logits

model = VisAnomReasoner().to(device)
print("VisAnomReasoner Architecture Loaded Successfully.")

VisAnomReasoner Architecture Loaded Successfully.


In [16]:
# [CELL 5 - Multi-Task Training Loop]

def build_gt_heatmap(gt_bounds, seq_tokens=50):
    """Generates ground-truth binary mask across the 50 temporal bins."""
    bin_centers = (torch.arange(seq_tokens, device=gt_bounds.device).float() + 0.5) / seq_tokens
    bin_centers = bin_centers.unsqueeze(0)  # [1, 50]

    gt_start = gt_bounds[:, 0:1]            # [B, 1]
    gt_end   = gt_bounds[:, 1:2]            # [B, 1]

    gt_heatmap = (bin_centers >= gt_start) & (bin_centers <= gt_end)
    return gt_heatmap.float()

def compute_1d_iou_loss(pred_bounds, true_bounds):
    """Differentiable 1D IoU Loss."""
    pred_start, pred_end = pred_bounds[:, 0], pred_bounds[:, 1]
    true_start, true_end = true_bounds[:, 0], true_bounds[:, 1]

    inter_start = torch.maximum(pred_start, true_start)
    inter_end   = torch.minimum(pred_end, true_end)
    intersection = torch.clamp(inter_end - inter_start, min=0.0)

    union = (pred_end - pred_start) + (true_end - true_start) - intersection + 1e-6
    iou = intersection / union
    return iou, 1.0 - iou.mean()

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
bce_criterion = nn.BCEWithLogitsLoss()
l1_criterion = nn.SmoothL1Loss()

epochs = 25
print("Starting VisAnomReasoner Fine-Tuning on VisAnomBench...")

for epoch in range(1, epochs + 1):
    model.train()
    total_loss, total_iou = 0.0, 0.0

    for batch in train_loader:
        images = batch["image"].to(device)
        gt_bounds = batch["bounds"].to(device)

        optimizer.zero_grad()
        pred_bounds, heatmap_logits = model(images)

        # 1. Auxiliary Dense Heatmap Loss
        gt_heatmap = build_gt_heatmap(gt_bounds, seq_tokens=50)
        loss_heatmap = bce_criterion(heatmap_logits, gt_heatmap)

        # 2. Continuous Bounds Regression & 1D IoU Loss
        batch_ious, loss_iou = compute_1d_iou_loss(pred_bounds, gt_bounds)
        loss_l1 = l1_criterion(pred_bounds, gt_bounds)

        loss = loss_heatmap + loss_l1 + 2.0 * loss_iou
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_iou += batch_ious.mean().item()

    if epoch % 5 == 0 or epoch == 1:
        avg_loss = total_loss / len(train_loader)
        avg_iou = total_iou / len(train_loader)
        print(f"Epoch [{epoch:02d}/{epochs}] | Loss: {avg_loss:.4f} | Interval IoU: {avg_iou*100:.2f}%")

Starting VisAnomReasoner Fine-Tuning on VisAnomBench...
Epoch [01/25] | Loss: 2.4120 | Interval IoU: 1.24%
Epoch [05/25] | Loss: 2.2962 | Interval IoU: 1.18%
Epoch [10/25] | Loss: 2.1744 | Interval IoU: 1.45%
Epoch [15/25] | Loss: 2.1170 | Interval IoU: 1.45%
Epoch [20/25] | Loss: 2.0962 | Interval IoU: 1.45%
Epoch [25/25] | Loss: 2.0873 | Interval IoU: 1.45%


In [14]:
# [CELL 6 - Evaluation Benchmark]

def evaluate_visanom():
    model.eval()
    all_preds, all_gts = [], []

    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(device)
            pred_bounds, _ = model(images)
            all_preds.append(pred_bounds.cpu().numpy())
            all_gts.append(batch["bounds"].numpy())

    all_preds = np.vstack(all_preds)
    all_gts   = np.vstack(all_gts)

    pred_indices = (all_preds * 200).astype(int)
    gt_indices   = (all_gts * 200).astype(int)

    iforest_ious = []
    reasoner_ious = []

    for i in range(len(test_dataset)):
        sample = test_dataset[i]
        sig = sample["signal"].numpy().reshape(-1, 1)
        gt_s, gt_e = sample["raw_bounds"]

        # Isolation Forest Baseline
        clf = IsolationForest(contamination=0.1, random_state=42)
        preds = clf.fit_predict(sig)
        anom_idxs = np.where(preds == -1)[0]

        if len(anom_idxs) > 0:
            if_s, if_e = anom_idxs.min(), anom_idxs.max()
        else:
            if_s, if_e = 0, 0

        inter = max(0, min(if_e, gt_e) - max(if_s, gt_s))
        union = max(1, max(if_e, gt_e) - min(if_s, gt_s))
        iforest_ious.append(inter / union)

        # VisAnomReasoner (Ours)
        pr_s, pr_e = pred_indices[i]
        r_inter = max(0, min(pr_e, gt_e) - max(pr_s, gt_s))
        r_union = max(1, max(pr_e, gt_e) - min(pr_s, gt_s))
        reasoner_ious.append(r_inter / r_union)

    print("\n" + "=" * 60)
    print("BENCHMARK EVALUATION RESULTS (VisAnomBench Test Set)")
    print("=" * 60)
    print(f"Isolation Forest Baseline Mean Interval IoU : {np.mean(iforest_ious)*100:.2f}%")
    print(f"VisAnomReasoner (Ours)    Mean Interval IoU : {np.mean(reasoner_ious)*100:.2f}%")
    print("=" * 60)
    print(f"Net Improvement over Baseline              : +{(np.mean(reasoner_ious) - np.mean(iforest_ious))*100:.2f} pp")

evaluate_visanom()


BENCHMARK EVALUATION RESULTS (VisAnomBench Test Set)
Isolation Forest Baseline Mean Interval IoU : 15.54%
VisAnomReasoner (Ours)    Mean Interval IoU : 10.26%
Net Improvement over Baseline              : +-5.28 pp
